In [41]:
import pandas as pd
import numpy as np
import datetime

from pathlib import Path 

In [42]:
current_dir = Path(Path.cwd()).parent
data_dir = current_dir / "data"
source_data_dir = data_dir / "source"
print(data_dir)

c:\Users\pedro\DEV\dengue_prediction\dengue_prediction\data


In [43]:

import re
import unicodedata
 
def show_columns(df, n_range = 7):
    df = list(df.columns)
    l = []
    range = n_range
    for c in df:
        if range == n_range:
            print(l)
            l = []
            range = 0
        range += 1
        l.append(c)
    print("\n")

def normalize_column_name(col):
    # tira acentos
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    # minúsculas
    col = col.lower()
    # troca qualquer coisa que não seja letra ou número por _
    col = re.sub(r"[^a-z0-9]+", "_", col)
    # remove _ no começo e no fim
    col = col.strip("_")
    return col

In [44]:

def prep_DEGBR(df:pd.DataFrame, ano:int, df_cod_municip:pd.DataFrame):
    # id municipio para municipio e UF
    # id_municip -> códigos IBGE
    df = df.merge(
        df_cod_municip,
        left_on="ID_MUNICIP",
        right_on="codigo_uf",
        how="left"
    )
    # Normalizacao colunas 
    df.columns = [normalize_column_name(col) for col in df.columns]

def get_municip_map_df(cod_munipc_uniques):
    # load csv
    municipios = pd.read_csv(
        source_data_dir / "municipios.csv",
        usecols=["codigo_ibge", "nome", "codigo_uf"],
        sep=","
    )
    municipios["codigo_municipio"] = municipios["codigo_ibge"] // 10
    # load csv
    estados = pd.read_csv(
        source_data_dir / "estados.csv",
        usecols=["codigo_uf", "uf"],
        sep=","
    )
    estados["codigo_municipio"] = estados["codigo_uf"] // 10
    
    municipios = municipios.merge(
        estados,
        on="codigo_uf",
        how="left"
    )
    municipios.drop(columns=["codigo_uf"])
    municipios_filtrados = municipios[
        municipios["codigo_ibge"].isin(cod_munipc_uniques)
    ].copy()
    return municipios_filtrados

In [ ]:
dengBr_dir = source_data_dir / "DENGBR" 
dengBr23_file = dengBr_dir / "DENGBR23.csv"
dengBr24_file = dengBr_dir / "DENGBR24.csv"
dengBr25_file = dengBr_dir / "DENGBR25.csv"

dengBr23 = pd.read_csv(dengBr23_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")
dengBr24 = pd.read_csv(dengBr24_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")
dengBr25 = pd.read_csv(dengBr25_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")

cod_munipc_uniques = dengBr23["ID_MUNICIP"].unique().tolist()
cod_munipc_uniques.extend(dengBr24["ID_MUNICIP"].unique().tolist())
cod_munipc_uniques.extend(dengBr25["ID_MUNICIP"].unique().tolist())

df_cod_municip = get_municip_map_df(cod_munipc_uniques)



#sprep_DEGBR(dengBr23, 23, df_cod_municip)
#prep_DEGBR(dengBr24, 24, df_cod_municip)
#prep_DEGBR(dengBr25, 25, df_cod_municip)
